<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter10/10_02_taylor_maccoll_cone_sweep.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Cone Shock-Angle Search and Taylor-Maccoll Property Sweep

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 10:** Conical Flow<br>
**Concepts:** cone-angle/shock-angle relation · event-driven ODE integration · surface Mach number · surface pressure

## Learning objectives

- Search for the shock angle associated with a prescribed cone half-angle.
- Integrate the Taylor-Maccoll equations with a cone-surface event.
- Map free-stream Mach number to surface Mach and pressure ratio.

## How to use this notebook

The first cell defines an optional interactive shock-angle tool. The next cell performs the deterministic 10-degree cone sweep; the last cell saves its two figures under `outputs/`.

> **Model scope.** The beta sweep is a transparent teaching search rather than an optimized production solver; refine the sweep and ODE tolerances before using it as reference data.


In [ ]:
import numpy as np

def cotan(x):
    """Compute cotangent of x"""
    return 1.0 / np.tan(x)

def conical_flow():
    gamma = 1.4
    pi = np.pi
    n = 2000  # Number of integration steps

    # Output file for cone angle vs shock angle
    with open("teta_beta_mach_Diagram.plt", "w", encoding="utf-8") as f:
        f.write("Variables=Semi_Cone_Angle_(deg),Shock_Angle_(deg)\n")

        # Free-stream Mach number
        M_inf = float(input("Enter Free Stream Mach Number: "))

        condition = 1
        while condition > 0:

            # Cone semi-angle in degrees
            t_c = float(input("Enter Cone Semi-angle (in degrees): "))
            theta_c = np.deg2rad(t_c)

            # Initial guess for shock angle (radians)
            theta_s = (60.0 / M_inf / 180.0) * pi

            # Allocate arrays
            F = np.zeros(n)
            G = np.zeros(n)
            H = np.zeros(n)
            th = np.zeros(n)

            kk = 1
            er = 0.1  # Initial error value

            # Iterative loop for shock angle convergence
            while abs(er) > 1e-5:

                # Flow angle immediately behind the shock
                tetha_w = np.arctan(
                    2.0 * cotan(theta_s)
                    * ((M_inf * np.sin(theta_s))**2 - 1.0)
                    / (M_inf**2 * (gamma + np.cos(2.0 * theta_s)) + 2.0)
                )

                # Normal Mach number before the shock
                M_n1 = M_inf * np.sin(theta_s)

                # Normal Mach number after the shock
                M_n2 = np.sqrt(
                    (1.0 + (gamma - 1.0) / 2.0 * M_n1**2)
                    / (gamma * M_n1**2 - (gamma - 1.0) / 2.0)
                )

                # Mach number immediately behind the shock
                M2 = M_n2 / np.sin(theta_s - tetha_w)

                # Non-dimensional velocity magnitude behind the shock
                V_pr = 1.0 / np.sqrt(2.0 / (gamma - 1.0) / M2**2 + 1.0)

                # Radial and tangential velocity components
                F[0] = V_pr * np.cos(tetha_w - theta_s)
                G[0] = V_pr * np.sin(tetha_w - theta_s)

                # Initial value of H
                H[0] = (
                    ((gamma - 1.0) / 2.0 * (-1.0 + F[0]**2 + G[0]**2)
                     * (2.0 * F[0] + G[0] * cotan(theta_s))
                     + F[0] * G[0]**2)
                    / ((gamma - 1.0) / 2.0 * (1.0 - F[0]**2 - G[0]**2)
                       - G[0]**2)
                )

                # Angular step size
                dt = (-theta_s + theta_c) / (n - 1)
                th[0] = theta_s

                # Numerical integration from shock to cone surface
                for i in range(n - 1):
                    th[i + 1] = th[i] + dt
                    F[i + 1] = F[i] + G[i] * dt + 0.5 * H[i] * dt**2
                    G[i + 1] = G[i] + H[i] * dt

                    numerator = (
                        (gamma - 1.0) / 2.0 * (-1.0 + F[i + 1]**2 + G[i + 1]**2)
                        * (2.0 * F[i + 1] + G[i + 1] * cotan(th[i + 1]))
                        + F[i + 1] * G[i + 1]**2
                    )
                    denominator = (
                        (gamma - 1.0) / 2.0 * (1.0 - F[i + 1]**2 - G[i + 1]**2)
                        - G[i + 1]**2
                    )
                    H[i + 1] = numerator / denominator

                # Error defined as tangential velocity at cone surface
                er = G[-1]

                # Shock angle correction formula
                theta_s = theta_s - 0.006 * t_c * er

                kk += 1
                print("Iteration:", kk)

            # Final shock angle in degrees
            shock_angle_deg = np.rad2deg(theta_s)
            print("Shock angle is:", shock_angle_deg, "Deg")

            # Write results to file
            f.write(f"{t_c} {shock_angle_deg}\n")

            # User choice for next run
            Quest = int(input(
                "If you want to try another cone angle with the same Mach number press 1, otherwise press 0: "
            ))
            condition = 1 if Quest == 1 else 0

    input("Press Enter to exit...")

# Optional interactive use: call conical_flow().


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

gamma = 1.4
theta_target_deg = 10
theta_target = np.radians(theta_target_deg)

def taylor_maccoll(theta, y):
    Vr, Vtheta = y

    a2 = 0.5*(gamma-1)*(1 - Vr**2 - Vtheta**2)

    dVr = Vtheta

    den = Vtheta**2 - a2

    if abs(den) < 1e-8:
        den = np.copysign(1e-8, den if den != 0 else 1.0)

    dVtheta = -Vr + (a2*(Vr + Vtheta/np.tan(theta))) / den

    return [dVr, dVtheta]


def surface_event(theta, y):
    return y[1]

surface_event.terminal = True
surface_event.direction = 1


M1_range = np.linspace(1.1, 3, 20)

results = []

for M1 in M1_range:

    mach_angle = np.arcsin(1/M1)

    beta_sweep = np.linspace(
        mach_angle + 1e-4,
        np.pi/2 - 1e-4,
        160
    )

    best_error = 1e9
    best = None

    for beta in beta_sweep:

        M1n = M1 * np.sin(beta)
        Mn1_sq = M1n**2

        rho_ratio = ((gamma+1)*Mn1_sq) / ((gamma-1)*Mn1_sq + 2)

        V1_Vmax = np.sqrt(
            M1**2 / (2/(gamma-1) + M1**2)
        )

        Vr_beta = V1_Vmax * np.cos(beta)

        Vtheta_beta = -V1_Vmax * np.sin(beta) / rho_ratio

        numerator = M1**2*np.sin(beta)**2 - 1

        denominator = M1**2*(gamma + np.cos(2*beta)) + 2

        delta = np.arctan(
            2*(1/np.tan(beta))*numerator/denominator
        )

        Mn2 = np.sqrt(
            (1 + 0.5*(gamma-1)*Mn1_sq) /
            (gamma*Mn1_sq - 0.5*(gamma-1))
        )

        M2 = Mn2 / np.sin(beta - delta)

        try:

            res = solve_ivp(
                taylor_maccoll,
                [beta, 1e-3],
                [Vr_beta, Vtheta_beta],
                events=surface_event,
                rtol=1e-9,
                atol=1e-9,
                max_step=1e-3
            )

            if res.status == 1:

                theta_surface = res.t_events[0][0]

                error = abs(theta_surface - theta_target)

                if error < best_error:

                    Vr_surf = res.y[0, -1]
                    Vtheta_surf = res.y[1, -1]

                    a2 = 0.5*(gamma-1)*(1 - Vr_surf**2 - Vtheta_surf**2)

                    a_surf = np.sqrt(a2)

                    M_surf = Vr_surf / a_surf

                    term1 = 1 + (2*gamma/(gamma+1))*(M1**2*np.sin(beta)**2 - 1)

                    term2 = 1 + (gamma-1)/2 * M2**2

                    term3 = 1 + (gamma-1)/2 * M_surf**2

                    pressure_ratio = term1*((term2/term3)**(gamma/(gamma-1)))

                    best_error = error

                    best = [M1, M_surf, pressure_ratio]

        except (FloatingPointError, ValueError, ZeroDivisionError):
            pass

    if best:
        results.append(best)


M1_values = [r[0] for r in results]
M_surface_values = [r[1] for r in results]
pressure_values = [r[2] for r in results]


plt.figure(figsize=(8,5))

plt.plot(
    M1_values,
    M_surface_values,
    'bo-',
    linewidth=2
)

plt.xlabel('Free Stream Mach Number M1')
plt.ylabel('Surface Mach Number')
plt.title('Surface Mach Number vs Free Stream Mach Number')
plt.grid(True)

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    M1_values,
    pressure_values,
    'rs-',
    linewidth=2
)

plt.xlabel('Free Stream Mach Number M1')
plt.ylabel('Pressure Ratio ps/p1')
plt.title('Pressure Ratio vs Free Stream Mach Number')
plt.grid(True)

plt.show()


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

plot1 = output_dir / "mach_vs_surface.png"
plt.figure(figsize=(8, 5))
plt.plot(M1_values, M_surface_values, "bo-", linewidth=2)
plt.xlabel("Free-stream Mach number M1")
plt.ylabel("Surface Mach number")
plt.title("Surface Mach Number vs Free-stream Mach Number")
plt.grid(True)
plt.savefig(plot1, dpi=300, bbox_inches="tight")
plt.show()

plot2 = output_dir / "pressure_ratio.png"
plt.figure(figsize=(8, 5))
plt.plot(M1_values, pressure_values, "rs-", linewidth=2)
plt.xlabel("Free-stream Mach number M1")
plt.ylabel("Pressure ratio ps/p1")
plt.title("Surface Pressure Ratio vs Free-stream Mach Number")
plt.grid(True)
plt.savefig(plot2, dpi=300, bbox_inches="tight")
plt.show()

print(plot1.resolve())
print(plot2.resolve())


## Interpretation and verification checklist

- Repeat the sweep with a finer beta grid and verify that the selected curves are stable.
- Reject states with nonpositive nondimensional sound-speed squared.
- Record cone angle, gamma, sweep resolution, and ODE tolerances with any reported curve.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
